In [38]:
import spacy
import re
from spacy.pipeline import EntityRuler
from spacy.matcher import Matcher
from collections import defaultdict
from collections import OrderedDict


nlp = spacy.load("en_core_web_sm")

# -------- EntityRuler for days and times --------
ruler = nlp.add_pipe("entity_ruler", before="ner")
day_abbrs = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
special_days = ["Public Holiday", "Day before public holiday", "Day after public holiday"]

patterns = [
    *[{"label": "DAY", "pattern": [{"LOWER": d.lower()}]} for d in day_abbrs],
    *[{"label": "SPECIAL_DAY", "pattern": [{"LOWER": t.lower()} for t in day.split()]} for day in special_days],
    {"label": "TIME", "pattern": [{"TEXT": {"REGEX": r"\d{1,2}:\d{2}"}}]}
]
ruler.add_patterns(patterns)

# -------- Matcher for L.O. --------
matcher = Matcher(nlp.vocab)
matcher.add("LAST_ORDER", [[
    {"TEXT": {"REGEX": r"(?i)^l\.?o\.?$"}},
    {"TEXT": {"REGEX": r"^\d{1,2}:\d{2}$"}}
]])

# -------- Mappings --------
day_map = {
    "Mon": "Monday", "Tue": "Tuesday", "Wed": "Wednesday",
    "Thu": "Thursday", "Fri": "Friday", "Sat": "Saturday", "Sun": "Sunday",
    "Public Holiday": "public_holiday",
    "Day before public holiday": "day_before_public_holiday",
    "Day after public holiday": "day_after_public_holiday"
}

# -------- Time parser for each row --------
def parse_single_time_block(dtl_text):
    doc = nlp(dtl_text)
    matches = matcher(doc)

    # Collect L.O. times first (don't let them be mistaken as open/close times)
    lo_times = set()
    for match_id, start, end in matches:
        lo_times.add(doc[end - 1].text)

    # Now collect open/close times — filter out the L.O. times
    times = [ent.text for ent in doc.ents if ent.label_ == "TIME" and ent.text not in lo_times]
    last_orders = list(lo_times)

    time_blocks = []
    for i in range(0, len(times), 2):
        block = OrderedDict()
        block["open"] = times[i]
        block["close"] = times[i + 1] if i + 1 < len(times) else None
        block["last_order"] = last_orders[i // 2] if i // 2 < len(last_orders) else None
        time_blocks.append(block)

    return time_blocks

def group_days(day_blocks):
    """Group consecutive days into ranges: e.g., ['Monday', 'Tuesday', 'Wednesday'] → 'Mon–Wed'"""
    order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    abbrev = {'Monday': 'Mon', 'Tuesday': 'Tue', 'Wednesday': 'Wed',
              'Thursday': 'Thu', 'Friday': 'Fri', 'Saturday': 'Sat', 'Sunday': 'Sun'}

    ranges = []
    days = sorted(day_blocks, key=lambda d: order.index(d))
    current_range = [days[0]]

    for i in range(1, len(days)):
        prev = order.index(current_range[-1])
        curr = order.index(days[i])
        if curr == prev + 1:
            current_range.append(days[i])
        else:
            if len(current_range) == 1:
                ranges.append(abbrev[current_range[0]])
            else:
                ranges.append(f"{abbrev[current_range[0]]}–{abbrev[current_range[-1]]}")
            current_range = [days[i]]

    if current_range:
        if len(current_range) == 1:
            ranges.append(abbrev[current_range[0]])
        else:
            ranges.append(f"{abbrev[current_range[0]]}–{abbrev[current_range[-1]]}")

    return ", ".join(ranges)


def extract_new_year_holidays(closed_on_text):
    """
    Extracts New Year holiday dates from freeform text like:
    'Year-end & New Year Holidays (December 30, 31 and January 1)'
    """
    if not closed_on_text:
        return []

    if "new year" in closed_on_text.lower():
        match = re.search(r"\((.*?)\)", closed_on_text)
        if match:
            inner_text = match.group(1)
            inner_text = inner_text.replace(" and ", ", ")
            date_tokens = [d.strip() for d in inner_text.split(",")]

            normalized = []
            current_month = ""
            for token in date_tokens:
                parts = token.split()
                if len(parts) == 2:
                    current_month = parts[0]
                    normalized.append(f"{parts[0]} {parts[1]}")
                elif len(parts) == 1 and current_month:
                    normalized.append(f"{current_month} {parts[0]}")
            return normalized
    return []

# -------- Master parser for all hours_raw --------
def parse_hours_raw(hours_raw, closed_on_text=""):
    schedule = defaultdict(list)

    for entry in hours_raw:
        title = entry.get("title", "")
        dtl = entry.get("dtl", "")

        if not title:
            continue  # skip rows like {"list_context": "other"}

        # Normalize day labels
        raw_days = [d.strip() for d in title.split(",")]
        days = [day_map.get(d, d) for d in raw_days]

        # Handle closed days
        if "closed" in dtl.lower():
            for day in days:
                schedule[day] = "closed"
            continue

        # Extract open/close blocks
        blocks = parse_single_time_block(dtl)
        for day in days:
            schedule[day].extend(blocks)

    # Start final output
    result = dict(schedule)

    # Add holiday closures
    new_year_days = extract_new_year_holidays(closed_on_text)
    if new_year_days:
        result["holiday_closures"] = {"new_year": new_year_days}

    # Add text summary
    result["text_summary"] = format_text_summary(schedule)

    # Add notes if available
    for entry in hours_raw:
        notice = entry.get("open_closed_notice")
        if notice:
            result["notes"] = notice
            break

    return result



def format_text_summary(schedule):
    """Generate readable text summary like 'Mon–Sun: 16:00–01:00 (L.O. 00:30)'"""
    summary_blocks = defaultdict(list)

    for day, blocks in schedule.items():
        if blocks == "closed":
            continue
        for block in blocks:
            key = f"{block['open']}–{block['close']}"
            if block.get("last_order"):
                key += f" (L.O. {block['last_order']})"
            summary_blocks[key].append(day)

    summary_lines = []
    for time_range, days in summary_blocks.items():
        summary_lines.append(f"{group_days(days)}: {time_range}")

    return "; ".join(summary_lines)

In [ ]:
##Test

restaurant_data = {
    "hours_raw": [
        {
        "list_context": "main",
        "title": "Mon, Tue, Wed, Thu, Fri, Sat",
        "dtl": "18:00 - 01:00",
        "dtl_text": "18:00 - 01:00",
        "open_closed_notice": "Hours and closed days may change, so please check with the restaurant before visiting."
      },
      {
        "list_context": "main",
        "title": "Sun",
        "dtl": "Closed",
        "dtl_text": "Closed",
        "open_closed_notice": "Hours and closed days may change, so please check with the restaurant before visiting."
      },
      {
        "list_context": "other",
        "open_closed_notice": "Hours and closed days may change, so please check with the restaurant before visiting."
      }
    ],
    "hours_notes_structured": {
      "business_hours": {},
      "closed_on": "The third Monday of each month"
    }
}


In [42]:
parsed = parse_hours_raw(
    restaurant_data["hours_raw"],
    restaurant_data["hours_notes_structured"].get("closed_on", "")
)

from pprint import pprint
pprint(parsed)


{'Friday': [OrderedDict([('open', '18:00'),
                         ('close', '01:00'),
                         ('last_order', None)])],
 'Monday': [OrderedDict([('open', '18:00'),
                         ('close', '01:00'),
                         ('last_order', None)])],
 'Saturday': [OrderedDict([('open', '18:00'),
                           ('close', '01:00'),
                           ('last_order', None)])],
 'Sunday': 'closed',
 'Thursday': [OrderedDict([('open', '18:00'),
                           ('close', '01:00'),
                           ('last_order', None)])],
 'Tuesday': [OrderedDict([('open', '18:00'),
                          ('close', '01:00'),
                          ('last_order', None)])],
 'Wednesday': [OrderedDict([('open', '18:00'),
                            ('close', '01:00'),
                            ('last_order', None)])],
 'notes': 'Hours and closed days may change, so please check with the '
          'restaurant before visiting.',
 'text_s